# Income and Sex in the UCI Adult Data: A Statistical Analysis

**Analytical question.** In this dataset, is sex associated with a person’s higher-income target, and how strong is the association?

UCI Adult (Census Income) data drives this analysis. The notebook summarizes the table, compares the income target by sex, investigates the main patterns with figures, and uses a chi-square test of independence. Every value used in the interpretation is printed by a cell.


## Setup

First, I import the analysis libraries and print the versions used by this run. I also set one visible style for every figure so later cells cannot change the display behavior silently.


In [1]:
# Core libraries for the analysis and the figures
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", None)
os.makedirs("figures", exist_ok=True)  # the four figure cells write here

print("numpy", np.__version__, "| pandas", pd.__version__, "| matplotlib", matplotlib.__version__)
print("seaborn", sns.__version__, "| scipy", scipy.__version__)

numpy 2.5.3 | pandas 3.0.6 | matplotlib 3.11.2
seaborn 0.13.2 | scipy 1.18.1


## Data ingestion

The dataset file sits in the same folder as this notebook. I load it, confirm the shape and the first rows, and check the types, the missing values, and the duplicates before any analysis.

Source: UCI Machine Learning Repository, Adult (Census Income), donated by Barry Becker and Ronny Kohavi (1996).

In [2]:
# Load the dataset that sits beside this notebook
DATA_PATH = "adult.csv"
df = pd.read_csv(DATA_PATH, na_values="?")

print("Shape:", df.shape)
df.head()

Shape: (32561, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [3]:
print(df.dtypes.to_string())
print()
print("Total missing values:", int(df.isnull().sum().sum()))
print("Columns with missing values:", df.columns[df.isnull().any()].tolist())
print("Exact duplicate rows:", int(df.duplicated().sum()))

age               int64
workclass           str
fnlwgt            int64
education           str
education-num     int64
marital-status      str
occupation          str
relationship        str
race                str
sex                 str
capital-gain      int64
capital-loss      int64
hours-per-week    int64
native-country      str
income              str

Total missing values: 4262
Columns with missing values: ['workclass', 'occupation', 'native-country']
Exact duplicate rows: 24


The file loads cleanly. There are `15` columns: `age`, `fnlwgt`, `education-num`, `capital-gain`, `capital-loss`, and `hours-per-week` are numeric, and the rest are text categories. Three columns have missing values (`workclass` `1,836`, `occupation` `1,843`, `native-country` `583`); the income and sex columns are complete, so the test below is unaffected. There are `24` exact duplicate rows, which I keep: a census extract can legitimately hold two people with identical attributes, so dropping them would remove real rows.